# 04. Preprocesamiento de Features de Contenido

Este notebook transforma el enriquecimiento IMDb + MovieLens en un conjunto de features reutilizable para híbrido y deep learning.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from collections import Counter

from src.utils.recsys_utils import build_movies_enriched, ensure_dir, flatten_column, load_clean_data, prepare_content_features

imdb, movielens = load_clean_data()
processed_dir = ensure_dir(ROOT / 'data' / 'processed')
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')
items_master = build_movies_enriched(imdb, movielens)


## Construir tabla maestra de ítems

Reutilizamos el join del notebook base y agregamos señales de personas para reducir el trabajo repetido en modelado.


In [ ]:
item_quality = pd.DataFrame([
    {'metric': 'items_master_rows', 'value': len(items_master)},
    {'metric': 'with_directors', 'value': items_master['directors_list'].apply(len).gt(0).mean()},
    {'metric': 'with_writers', 'value': items_master['writers_list'].apply(len).gt(0).mean()},
    {'metric': 'with_principals', 'value': items_master['principal_names'].apply(len).gt(0).mean()},
])
display(item_quality)


## Crear features estructuradas

Generamos variables numéricas, one-hot de `titleType` y bolsas controladas para directores y cast principal.


In [ ]:
item_features = prepare_content_features(items_master, movielens['tags'])
item_features = item_features.merge(
    pd.get_dummies(items_master[['movieId', 'titleType']], columns=['titleType'], dummy_na=True),
    on='movieId',
    how='left'
)

def top_token_frame(series, prefix, top_k=50):
    token_counts = Counter(flatten_column(series))
    top_tokens = {token for token, _ in token_counts.most_common(top_k)}
    rows = []
    for values in series:
        current = set(values if isinstance(values, list) else [])
        rows.append({f'{prefix}_{token}': int(token in current) for token in top_tokens})
    return pd.DataFrame(rows)

director_features = top_token_frame(items_master['directors_list'], 'director', top_k=40)
principal_features = top_token_frame(items_master['principal_names'], 'principal', top_k=60)

item_features = pd.concat([
    item_features.reset_index(drop=True),
    director_features.reset_index(drop=True),
    principal_features.reset_index(drop=True),
], axis=1)

display(item_features.head())
print('Shape final de item_features:', item_features.shape)


## Guardar artefactos de contenido

Dejamos listos `items_master` y `item_features` para los notebooks de híbrido y deep learning.


In [ ]:
items_master.to_parquet(processed_dir / 'items_master.parquet', index=False)
item_features.to_parquet(processed_dir / 'item_features.parquet', index=False)
item_quality.to_csv(tables_dir / 'item_feature_quality.csv', index=False)
